# Getting started

In [ ]:
#!pip install neuralforecast

### Import libraries

In [ ]:
import neuralforecast as nf
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch as t
import matplotlib.pyplot as plt

### Dataset

In [ ]:
Y_df, _, S_df = nf.data.datasets.m5.M5.load('data')
Y_df['ds'] = pd.to_datetime(Y_df['ds'])
Y_df.head()

In [ ]:
Y_df = Y_df.merge(S_df, how='left', on=['unique_id'])

In [ ]:
Y_df = Y_df[['item_id', 'ds', 'y']].groupby(['item_id', 'ds']).sum().reset_index()

In [ ]:
Y_df = Y_df.rename(columns={'item_id': 'unique_id'})

In [ ]:
Y_df = Y_df.groupby('unique_id').tail(100)

In [ ]:
uids = Y_df['unique_id'].unique()[:100]
Y_df = Y_df.query('unique_id in @uids').reset_index(drop=True)
Y_df['unique_id'] = Y_df['unique_id'].astype(str)

In [ ]:
Y_df

### Split train/test sets

In [ ]:
output_size = 7
Y_df_test = Y_df.groupby('unique_id').tail(output_size)
Y_df_train = Y_df.drop(Y_df_test.index)

### Define WindowsDataset and TimeSeriesLoader

In [ ]:
input_size = 4 * output_size

In [ ]:
train_mask_df, val_mask_df, _ = nf.experiments.utils.get_mask_dfs(
    Y_df=Y_df_train,
    ds_in_val=3 * output_size,
    ds_in_test=0
)

In [ ]:
plot_df = Y_df_train.merge(
    train_mask_df.drop('available_mask', axis=1).rename(columns={'sample_mask': 'sample_mask_train'}),
    how='left',
    on=['unique_id', 'ds']
).merge(
    val_mask_df.drop('available_mask', axis=1).rename(columns={'sample_mask': 'sample_mask_val'}),
    how='left',
    on=['unique_id', 'ds']
)

In [ ]:
plot_df['y_train'] = np.where(plot_df['sample_mask_train'] == 1, plot_df['y'], np.nan)
plot_df['y_val'] = np.where(plot_df['sample_mask_val'] == 1, plot_df['y'], np.nan)

In [ ]:
plot_df.query('unique_id == "HOBBIES_1_001"').set_index('ds')[['y_train', 'y_val']].plot()

In [ ]:
train_dataset = nf.data.tsdataset.TimeSeriesDataset(
    Y_df=Y_df_train, 
    input_size=input_size,
    output_size=output_size,
    mask_df=train_mask_df
)

In [ ]:
val_dataset = nf.data.tsdataset.TimeSeriesDataset(
    Y_df=Y_df_train, 
    input_size=input_size,
    output_size=output_size,
    mask_df=val_mask_df
)

In [ ]:
train_loader = nf.data.tsloader.TimeSeriesLoader(
    train_dataset, batch_size=256, 
    shuffle=True
)

In [ ]:
val_loader = nf.data.tsloader.TimeSeriesLoader(
    val_dataset, 
    batch_size=1
)

### Define model

In [ ]:
model = nf.models.esrnn.mqesrnn.MQESRNN(
    n_series=train_dataset.n_series,
    input_size=input_size, output_size=output_size,
    n_x=0, 
    frequency='D', 
    cell_type='GRU',
    state_hsize=64,
    training_percentiles=[5, 20, 50, 80, 95],
    testing_percentiles=[5, 20, 50, 80, 95],
    loss='MQ'
)

### Train model with early stopping

In [ ]:
early_stopping = pl.callbacks.EarlyStopping(monitor="val_loss")

trainer = pl.Trainer(max_epochs=100,
                     gpus=-1 if t.cuda.is_available() else 0,
                     callbacks=[early_stopping])

trainer.fit(model, train_loader, val_loader)

In [ ]:
Y_df_forecast = model.forecast(Y_df_train)
Y_df_forecast.head()

In [ ]:
Y_df_plot = Y_df_test.merge(Y_df_forecast, how='left', on=['unique_id', 'ds'])

In [ ]:
for uid in Y_df_plot['unique_id'].unique()[:5]:
    df_plot = Y_df_plot.query('unique_id == @uid')
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_plot['ds'], df_plot['y'], label='Observed', color='black', linewidth=0.8)
    ax.plot(df_plot['ds'], df_plot['y_p50'], label='P50', color='#2187bb')
    ax.fill_between(df_plot['ds'], df_plot['y_p20'], df_plot['y_p80'], color='#2187bb', alpha=.15, label='P20-P80')
    ax.fill_between(df_plot['ds'], df_plot['y_p5'], df_plot['y_p95'], color='#2187bb', alpha=.10, label='P5-P95')
    ax.legend()
    